In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adadelta

def attention_bottleneck(x):
    """
    Implementación del módulo de atención convolucional (Figura 3).
    Utiliza convoluciones 1x1 para generar una máscara de recalibración.
    """
    shortcut = x  # Entrada de 32x32x256
    
    # Rama de atención: reducción y mapa sigmoide
    a = layers.Conv2D(64, (1, 1), padding='same', activation='relu')(x)
    a = layers.Conv2D(32, (1, 1), padding='same', activation='relu')(a)
    a = layers.Conv2D(1,  (1, 1), padding='same', activation='sigmoid')(a)
    
    # Recalibración multiplicativa y conexión residual
    # Multiplicamos el shortcut por la máscara y luego sumamos
    attended = layers.Multiply()([shortcut, a])
    # Ajustamos canales si es necesario antes de la suma final
    attended_proj = layers.Conv2D(256, (1, 1), padding='same')(attended)
    
    return layers.Add(name="bottleneck_output")([shortcut, attended_proj])

def build_cae_bladder_cancer(input_shape=(128, 128, 3)):
    inputs = Input(shape=input_shape, name="histological_patch")
    
    # --- ENCODER (f_phi) ---
    # Tres capas convolucionales 3x3 con stride progresivo
    x = layers.Conv2D(64, (3, 3), strides=1, padding='same', activation='relu')(inputs)  # 128x128x64
    x = layers.Conv2D(128, (3, 3), strides=2, padding='same', activation='relu')(x)      # 64x64x128
    x = layers.Conv2D(256, (3, 3), strides=2, padding='same', activation='relu')(x)      # 32x32x256
    
    # --- BOTTLENECK CON ATENCIÓN ---
    z_i = attention_bottleneck(x)
    
    # --- DECODER (g_theta) ---
    # Batch Normalization antes de las convoluciones transpuestas
    x = layers.BatchNormalization()(z_i)
    x = layers.Conv2DTranspose(128, (3, 3), strides=2, padding='same', activation='relu')(x)
    
    x = layers.BatchNormalization()(x)
    x = layers.Conv2DTranspose(64, (3, 3), strides=2, padding='same', activation='relu')(x)
    
    # Reconstrucción final (r_i) con activación sigmoide
    outputs = layers.Conv2DTranspose(3, (3, 3), strides=1, padding='same', activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="CAE_BladderCancer_Figure3")
    
    # Configuración de entrenamiento según sección 3.1
    model.compile(optimizer=Adadelta(learning_rate=0.5), loss='mse')
    
    return model

# Crear el modelo y mostrar el resumen
cae_model = build_cae_bladder_cancer()
cae_model.summary()

Model: "CAE_BladderCancer_Figure3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ histological_patch            │ (None, 128, 128, 3)       │               0 │ -                          │
│ (InputLayer)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 128, 128, 64)      │           1,792 │ histological_patch[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 64, 64, 128)       │          73,856 │ conv2d[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 32, 32, 256)       │         295,168 │ conv2d_1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 32, 32, 64)        │          16,448 │ conv2d_2[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 32, 32, 32)        │           2,080 │ conv2d_3[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_5 (Conv2D)             │ (None, 32, 32, 1)         │              33 │ conv2d_4[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multiply (Multiply)           │ (None, 32, 32, 256)       │               0 │ conv2d_2[0][0],            │
│                               │                           │                 │ conv2d_5[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_6 (Conv2D)             │ (None, 32, 32, 256)       │          65,792 │ multiply[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bottleneck_output (Add)       │ (None, 32, 32, 256)       │               0 │ conv2d_2[0][0],            │
│                               │                           │                 │ conv2d_6[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 32, 32, 256)       │           1,024 │ bottleneck_output[0][0]    │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_transpose              │ (None, 64, 64, 128)       │         295,040 │ batch_normalization[0][0]  │
│ (Conv2DTranspose)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 64, 64, 128)       │             512 │ conv2d_transpose[0][0]     │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_transpose_1            │ (None, 128, 128, 64)      │          73,792 │ batch_normalization_1[0][… │
│ (Conv2DTranspose)             │                           │               

 Total params: 827,268 (3.16 MB)

 Trainable params: 826,500 (3.15 MB)

 Non-trainable params: 768 (3.00 KB)